# COMP813 – New experiments (shortcut study)
Run this notebook **next to** `COMP813_Project_Local.ipynb`. Each section says when to run it.

| Step | What | GPU? | Time |
|---|---|---|---|
| 0 | Find folders (always run first) | no | seconds |
| F1 | *Optional:* download FathomNet in-situ jellyfish | no (internet) | 15–40 min |
| A | Build the class × background benchmark | no | 5–10 min |
| B | Re-score every model + last-layer retraining (DFR) | **yes** | 20–40 min |
| C | CLIP + DINOv2 probes | **yes** | 10–20 min |
| F2 | *Optional:* in-situ retrain (only if F1 gave ≥100 crops) | **yes** | 30–60 min |
| D | Statistics, tables, figures | no | 2–5 min |

**Rule:** only ONE GPU step at a time. While the seed runs (814/815) are training in the other notebook, you may run **F1** and **A** here (they don't use the GPU). Start **B** only after the training has finished.


> **Note.** Cell outputs are cleared in this public version (they contained local paths and dataset images). The result files of the full runs (local RTX 4050 GPU, seeds 813–815) are in `results_snapshot/`.

## 0. Find the folders (run first, every time you open this notebook)

In [ ]:
import os, sys, json, glob, subprocess, codecs, shutil, time, re
from pathlib import Path
from IPython.display import Image, Markdown, display

# ---- Change this ONLY if your main folder is somewhere else ---------------------------
BASE_DIR = None          # e.g. BASE_DIR = r"D:\Uni\COMP813"   (None = look on the Desktop)
# ----------------------------------------------------------------------------------------

def find_base():
    if BASE_DIR:
        return Path(BASE_DIR).expanduser()
    home = Path.home()
    desktops = [home / "Desktop", home / "OneDrive" / "Desktop", home / "OneDrive - AUT University" / "Desktop"]
    desktops += list(home.glob("OneDrive*/Desktop"))
    for d in desktops:
        p = d / "AUT" / "AI" / "COMP813"
        if p.exists():
            return p
    raise FileNotFoundError("Could not find Desktop/AUT/AI/COMP813. Set BASE_DIR in this cell.")

def child(parent, name):
    """Find a sub-folder ignoring upper/lower case."""
    for p in parent.iterdir():
        if p.is_dir() and p.name.lower() == name.lower():
            return p
    return None

BASE = find_base()

# code folder = the sub-folder that contains run_all.py (works for project_code / COMP813_Project_Code ...)
CODE_DIR = next((p for p in sorted(BASE.iterdir()) if p.is_dir() and (p / "run_all.py").exists()), None)
if CODE_DIR is None:                                   # maybe one level deeper (unzipped twice)
    CODE_DIR = next((r.parent for r in BASE.glob("*/*/run_all.py")), None)
if CODE_DIR is None:
    raise FileNotFoundError(f"run_all.py not found inside {BASE}. Folders there: {[p.name for p in BASE.iterdir()]}")

DATA_ROOT = child(BASE, "Task") or child(BASE, "Tasks")
if DATA_ROOT is None:
    raise FileNotFoundError(f"No 'Task' folder inside {BASE}")
OUT_DIR = child(BASE, "outputs") or (BASE / "outputs")
OUT_DIR.mkdir(exist_ok=True)

# The project scripts read these two environment variables
os.environ["DATA_ROOT"] = str(DATA_ROOT)
os.environ["OUT_DIR"] = str(OUT_DIR)

ENV = dict(os.environ, PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8", PYTHONUTF8="1",
           KMP_DUPLICATE_LIB_OK="TRUE",   # avoids the Windows "two OpenMP libraries" crash (common with Anaconda)
           PYTHONFAULTHANDLER="1")        # prints where a native crash happened

def run(*args):
    """Run a project script with THIS notebook's Python, inside the code folder, and stream its output."""
    cmd = [sys.executable, "-u", *map(str, args)]
    print("$", " ".join(cmd), f"   (in {CODE_DIR.name})\n")
    t0 = time.time()
    proc = subprocess.Popen(cmd, cwd=str(CODE_DIR), env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    dec = codecs.getincrementaldecoder("utf-8")(errors="replace")
    tail = ""
    try:
        while True:
            chunk = proc.stdout.read1(4096)
            if not chunk:
                break
            txt = dec.decode(chunk); tail = (tail + txt)[-2500:]
            print(txt, end="", flush=True)
    except KeyboardInterrupt:                         # the Stop button in Jupyter
        proc.terminate(); proc.wait()
        print("\n[stopped - run the cell again to continue; finished runs are skipped]")
        raise
    rc = proc.wait()
    if rc != 0:
        hint = ""
        if "3221226505" in tail or "3221225477" in tail:
            hint = ("\n\nHINT: this is a silent Windows crash (no Python error). Run the same cell again - finished runs are skipped. "
                    "If it crashes at the same place, run the diagnostic cell in section 3.\n")
        raise RuntimeError(f"Command failed with exit code {rc}.{hint} LAST OUTPUT OF THE SCRIPT:\n\n{tail}")
    m = (time.time() - t0) / 60
    print(f"\n[finished in {m:.1f} min = {m/60:.2f} h]")

print("Main folder :", BASE)
print("Code folder :", CODE_DIR)
print("Data (Task) :", DATA_ROOT)
print("Outputs     :", OUT_DIR)
print("Python      :", sys.executable)

In [ ]:
run("train.py", "--task", "binary", "--model", "resnet50", "--quick")

In [ ]:
run("dfr.py", "--quick")

In [ ]:
run("stats.py", "--demo")

## F1. *(Optional)* Download FathomNet in-situ jellyfish
Safe to run while the other notebook is training. If it fails or gives < 100 crops, skip F2 – the project is still complete.

In [ ]:
run("-m", "pip", "install", "-q", "fathomnet")
run("fathomnet_download.py")

## A. Build the class × background benchmark (no GPU)
Afterwards **look at the two figures**: the jellyfish masks must roughly follow the animals.

In [ ]:
run("group_benchmark.py")
from IPython.display import Image as _Img, display
for f in ["fig_group_examples.png", "fig_jelly_mask_check.png"]:
    display(_Img(filename=str(OUT_DIR / "extra" / f)))

## B. Re-score every model on the benchmark + last-layer retraining (DFR) — GPU
Start only when the seed runs in the other notebook are finished.

In [ ]:
run("dfr.py", "--skip_existing")

## C. Foundation models: CLIP and DINOv2 (zero-shot and linear probes) — GPU
DINOv2 (~350 MB) is downloaded automatically the first time.

In [ ]:
run("foundation_probe.py")

## F2. *(Optional)* In-situ experiment with FathomNet jellyfish — GPU
Only if F1 produced at least ~100 crops. Re-run step A first so the in-situ group is added.

In [ ]:
run("insitu_experiment.py")

## D. Statistics, tables and figures (no GPU)
Then re-run the old summary so the 3-seed mean ± std tables are refreshed.

In [ ]:
run("stats.py")
run("run_all.py", "--stages", "summary")
from IPython.display import Markdown
display(Markdown((OUT_DIR / "extra" / "extra_key_numbers.md").read_text(encoding="utf-8")))

In [ ]:
import time
last = OUT_DIR / "robustness" / "deep_binary_vit_b_16_pt_ft_aug-underwater_s815.csv"
print("Waiting for the training in the other notebook to finish ...")
while not last.exists():
    time.sleep(120)            # checks every 2 minutes, uses no GPU while waiting
time.sleep(90)                 # let the other notebook release the GPU
run("dfr.py", "--skip_existing")          # B  (~40 min)
run("foundation_probe.py")                # C  (~15 min)
run("insitu_experiment.py")               # F2 (~45 min)
run("stats.py")                           # D  (~5 min)
run("run_all.py", "--stages", "summary")  # refreshes the old tables with 3 seeds
print("ALL DONE")

In [ ]:
for seed in ["814", "815"]:
    for task in ["binary", "fine"]:
        for model in ["resnet50", "efficientnet_b0", "vit_b_16"]:
            for aug in ["standard", "underwater"]:
                run("train.py", "--task", task, "--model", model, "--aug", aug, "--seed", seed)
run("dfr.py", "--skip_existing")          # tests the new models on the background benchmark + DFR
run("insitu_experiment.py")               # FathomNet test, now with all 3 seeds
run("stats.py")                           # tables and figures (also fixes a small cost-analysis detail)
run("run_all.py", "--stages", "summary")  # refreshes the old tables
print("ALL DONE")

In [ ]:
run("stats.py"); 
run("report_assets.py")

In [ ]:
run("make_zip.py")